In [1]:
# Parameters
run_id = "130b4b0c-059c-4a33-880b-6e04c41ce18c"
artifacts_dir = "/home/adnoman/projects/aml_gan/AMLend2end/artifacts/runs/130b4b0c-059c-4a33-880b-6e04c41ce18c"
sample_size = None
epochs = None
threshold = None


# Create node embeddings feature groups.

Up until now we use feature engineering, feature store and model training to create node embedding. We will now materialise this as node embeddings feature group. This feature group will be used to train anomaly detection model.

![Feature Stores](./images/online_offline_fs.png)

---
**NOTE**: 

In real life scenarios financial transaction are dynamically evolving graphs. If live Transaction Monitoring System is based on graph or node embeddings then this will require 1st to update the graph and node representations after new transactions arrive. Recomputing entire graph for every newly arrived transaction will lead to unaxeptable delayes and even monitoring system failures. This problem  will be more sever if large amount of updates happen in a short time window.

Contact us at Logical Clocks and we will help you to setup end to end graph based deep anomaly detection live Transaction Monitoring Systems. 

---

## Query Model Repository for best node embeddings model

In [2]:
# Setup for local execution
import os
import json
import pandas as pd
import numpy as np

# Define paths
BASE_PATH = os.path.dirname(os.path.abspath("__file__"))
TRAINING_DATA_PATH = os.path.join(BASE_PATH, "training_data")
OUTPUT_PATH = os.path.join(BASE_PATH, "output")
MODELS_PATH = os.path.join(BASE_PATH, "models")
RESOURCES_PATH = os.path.join(BASE_PATH, "Resources")

print(f"Training data: {TRAINING_DATA_PATH}")
print(f"Output: {OUTPUT_PATH}")

Training data: /home/adnoman/projects/aml_gan/AMLend2end/training_data
Output: /home/adnoman/projects/aml_gan/AMLend2end/output


In [3]:
# Find the latest model directory
model_dirs = [d for d in os.listdir(MODELS_PATH) if d.startswith('node_embeddings_')]
if model_dirs:
    latest_model_dir = os.path.join(MODELS_PATH, sorted(model_dirs)[-1])
    print(f"Found model: {latest_model_dir}")
    
    # Load metadata
    with open(os.path.join(latest_model_dir, 'metadata.json'), 'r') as f:
        metadata = json.load(f)
    print(f"Model metrics: {metadata['metrics']}")
    print(f"Hyperparameters: {metadata['hyperparameters']}")
else:
    print("No model found! Run notebook 4 first.")

Found model: /home/adnoman/projects/aml_gan/AMLend2end/models/node_embeddings_e2366de4
Model metrics: {'accuracy': 0.8891156462585034}
Hyperparameters: {'walk_number': 2, 'walk_length': 2, 'emb_size': 32}


In [4]:
# Load node embeddings from notebook 4
embeddings_path = os.path.join(TRAINING_DATA_PATH, "node_embeddings.csv")
node_embeddings_df = pd.read_csv(embeddings_path)

print(f"Loaded embeddings shape: {node_embeddings_df.shape}")
node_embeddings_df.head()

Loaded embeddings shape: (7347, 33)


,node_id,emb_0,emb_1,emb_2,emb_3,emb_4,emb_5,emb_6,emb_7,emb_8,...,emb_22,emb_23,emb_24,emb_25,emb_26,emb_27,emb_28,emb_29,emb_30,emb_31
0,3aa9646b,0.006354,-0.020029,0.000121,0.001682,0.022640,0.022485,-0.022633,-0.018184,-0.013343,...,-0.005648,0.013926,0.027505,-0.010198,0.025302,-0.008416,0.009119,0.013350,0.022079,-0.029613
1,1e46e726,0.002481,-0.007734,0.013989,-0.021754,0.023876,0.024569,0.005226,-0.014404,-0.014850,...,0.009040,-0.014103,0.029731,-0.024120,0.001362,0.002191,-0.000155,-0.023380,0.027062,0.014415
2,49203bc3,0.024728,-0.005727,0.011389,-0.003658,0.005619,0.027766,0.028874,-0.025506,-0.024554,...,-0.026760,-0.002589,0.025417,0.011257,-0.031272,0.029479,0.019596,-0.007734,0.025463,0.013300
3,a74d1101,0.006074,-0.028726,-0.022943,0.030213,0.001226,-0.016350,-0.007838,-0.003526,-0.005832,...,-0.024862,-0.011728,-0.017198,-0.015983,0.005139,0.021491,-0.009442,0.014519,-0.013948,-0.029704
4,616d4505,-0.020141,-0.024803,-0.019287,-0.006866,-0.015812,0.008584,-0.000066,0.009143,-0.012181,...,-0.030604,-0.023667,0.026482,-0.003340,-0.031200,-0.025585,0.000395,-0.019562,-0.010542,0.011896


## Define model and load wights 

In [5]:
# Get embedding columns
emb_cols = [c for c in node_embeddings_df.columns if c.startswith('emb_')]
print(f"Embedding dimensions: {len(emb_cols)}")

# Preview embeddings
node_embeddings_df[['node_id'] + emb_cols[:5]].head()

Embedding dimensions: 32


,node_id,emb_0,emb_1,emb_2,emb_3,emb_4
0,3aa9646b,0.006354,-0.020029,0.000121,0.001682,0.022640
1,1e46e726,0.002481,-0.007734,0.013989,-0.021754,0.023876
2,49203bc3,0.024728,-0.005727,0.011389,-0.003658,0.005619
3,a74d1101,0.006074,-0.028726,-0.022943,0.030213,0.001226
4,616d4505,-0.020141,-0.024803,-0.019287,-0.006866,-0.015812


## connect hsfs library and get fs handle

In [6]:
# Load alert nodes to join with embeddings
alert_nodes_df = pd.read_csv(os.path.join(TRAINING_DATA_PATH, "alert_nodes_td.csv"))
print(f"Alert nodes: {len(alert_nodes_df)}")
print(f"SAR nodes: {alert_nodes_df['is_sar'].sum()}")

Alert nodes: 7347
SAR nodes: 816


### Get node and edge traininhg dataset objects 

In [7]:
# Create embedding array column (for compatibility with original format)
node_embeddings_df['embedding'] = node_embeddings_df[emb_cols].values.tolist()

# Rename node_id to id for consistency
node_embeddings_df = node_embeddings_df.rename(columns={'node_id': 'id'})

# Select final columns
node_embeddings_final = node_embeddings_df[['id', 'embedding']].copy()
print(f"Final embeddings shape: {node_embeddings_final.shape}")
node_embeddings_final.head()

Final embeddings shape: (7347, 2)


,id,embedding
0,3aa9646b,"[0.006354473, -0.020029036, 0.00012067986, 0.0..."
1,1e46e726,"[0.002480811, -0.007734282, 0.013989168, -0.02..."
2,49203bc3,"[0.024727674, -0.005726637, 0.011388668, -0.00..."
3,a74d1101,"[0.0060743825, -0.02872574, -0.022943288, 0.03..."
4,616d4505,"[-0.020140577, -0.024802586, -0.019286547, -0...."


### Read training datasets as pandas df 

In [8]:
# Join embeddings with alert nodes info
embeddings_with_labels = node_embeddings_df.merge(
    alert_nodes_df[['id', 'is_sar']], 
    on='id', 
    how='left'
)
embeddings_with_labels['is_sar'] = embeddings_with_labels['is_sar'].fillna(0).astype(int)

print(f"Embeddings with labels: {embeddings_with_labels.shape}")
print(f"SAR nodes in embeddings: {embeddings_with_labels['is_sar'].sum()}")

Embeddings with labels: (7347, 35)
SAR nodes in embeddings: 816


### Read hyperparamenter for graph embeddings

In [9]:
# Preview the data
print("Sample of embeddings with SAR labels:")
embeddings_with_labels[['id', 'is_sar'] + emb_cols[:3]].head(10)

Sample of embeddings with SAR labels:


,id,is_sar,emb_0,emb_1,emb_2
0,3aa9646b,0,0.006354,-0.020029,0.000121
1,1e46e726,0,0.002481,-0.007734,0.013989
2,49203bc3,0,0.024728,-0.005727,0.011389
3,a74d1101,1,0.006074,-0.028726,-0.022943
4,616d4505,0,-0.020141,-0.024803,-0.019287
5,99af2455,1,0.022169,0.024115,0.014472
6,39be1ea2,0,0.002133,0.028336,-0.029831
7,e7ec7bdb,1,0.008609,-0.001301,0.016248
8,e2e0d938,0,0.029390,-0.018993,-0.000744
9,afc399a9,0,0.020601,-0.024435,0.021572


### Construct stellargraph Graph object

In [10]:
# Statistics
print("Embedding statistics:")
print(f"  Total nodes: {len(embeddings_with_labels)}")
print(f"  SAR nodes (is_sar=1): {embeddings_with_labels['is_sar'].sum()}")
print(f"  Non-SAR nodes (is_sar=0): {(embeddings_with_labels['is_sar']==0).sum()}")
print(f"  Embedding dimensions: {len(emb_cols)}")

Embedding statistics:
  Total nodes: 7347
  SAR nodes (is_sar=1): 816
  Non-SAR nodes (is_sar=0): 6531
  Embedding dimensions: 32


### infer node embeddings

In [11]:
# Prepare final feature group data
# Keep id, all embedding columns, and is_sar
final_cols = ['id'] + emb_cols + ['is_sar']
node_embeddings_fg_df = embeddings_with_labels[final_cols].copy()

print(f"Feature group shape: {node_embeddings_fg_df.shape}")
node_embeddings_fg_df.head()

Feature group shape: (7347, 34)


,id,emb_0,emb_1,emb_2,emb_3,emb_4,emb_5,emb_6,emb_7,emb_8,...,emb_23,emb_24,emb_25,emb_26,emb_27,emb_28,emb_29,emb_30,emb_31,is_sar
0,3aa9646b,0.006354,-0.020029,0.000121,0.001682,0.022640,0.022485,-0.022633,-0.018184,-0.013343,...,0.013926,0.027505,-0.010198,0.025302,-0.008416,0.009119,0.013350,0.022079,-0.029613,0
1,1e46e726,0.002481,-0.007734,0.013989,-0.021754,0.023876,0.024569,0.005226,-0.014404,-0.014850,...,-0.014103,0.029731,-0.024120,0.001362,0.002191,-0.000155,-0.023380,0.027062,0.014415,0
2,49203bc3,0.024728,-0.005727,0.011389,-0.003658,0.005619,0.027766,0.028874,-0.025506,-0.024554,...,-0.002589,0.025417,0.011257,-0.031272,0.029479,0.019596,-0.007734,0.025463,0.013300,0
3,a74d1101,0.006074,-0.028726,-0.022943,0.030213,0.001226,-0.016350,-0.007838,-0.003526,-0.005832,...,-0.011728,-0.017198,-0.015983,0.005139,0.021491,-0.009442,0.014519,-0.013948,-0.029704,1
4,616d4505,-0.020141,-0.024803,-0.019287,-0.006866,-0.015812,0.008584,-0.000066,0.009143,-0.012181,...,-0.023667,0.026482,-0.003340,-0.031200,-0.025585,0.000395,-0.019562,-0.010542,0.011896,0


In [12]:
# Dummy cell - removed pyspark code

In [13]:
# Dummy cell - removed pyspark code

In [14]:
# Dummy cell - removed pyspark code

In [15]:
# Dummy cell - removed pyspark code

In [16]:
# Dummy cell - removed pyspark code

## Create embeddings feature group

In [17]:
# Save node embeddings feature group locally (replaces hsfs)
fg_path = os.path.join(OUTPUT_PATH, "node_embeddings_fg.parquet")
node_embeddings_fg_df.to_parquet(fg_path, index=False)
print(f"Saved node embeddings feature group to: {fg_path}")

# Also save as CSV for easier inspection
csv_path = os.path.join(OUTPUT_PATH, "node_embeddings_fg.csv")
node_embeddings_fg_df.to_csv(csv_path, index=False)
print(f"Saved CSV version to: {csv_path}")

Saved node embeddings feature group to: /home/adnoman/projects/aml_gan/AMLend2end/output/node_embeddings_fg.parquet


Saved CSV version to: /home/adnoman/projects/aml_gan/AMLend2end/output/node_embeddings_fg.csv


In [18]:
# Summary
print("=" * 50)
print("Node Embeddings Feature Group Created")
print("=" * 50)
print(f"Total nodes: {len(node_embeddings_fg_df)}")
print(f"Embedding dimensions: {len(emb_cols)}")
print(f"SAR nodes: {node_embeddings_fg_df['is_sar'].sum()}")
print(f"Non-SAR nodes: {(node_embeddings_fg_df['is_sar']==0).sum()}")
print(f"\nSaved to: {fg_path}")
print("=" * 50)

Node Embeddings Feature Group Created
Total nodes: 7347
Embedding dimensions: 32
SAR nodes: 816
Non-SAR nodes: 6531

Saved to: /home/adnoman/projects/aml_gan/AMLend2end/output/node_embeddings_fg.parquet


## Feature group provenance
![Feature group provenance](./images/provenance_fg.png)

In [19]:
# Done!